# 02 RAG Setup

This notebook verifies that the knowledge base contains the starter documents plus at least five additional energy-saving documents. It then creates or loads the ChromaDB vector store used by the `search_energy_tips` tool.

In [1]:
from pathlib import Path
docs = sorted(Path('data/documents').glob('*.txt'))
print('Knowledge base documents:', len(docs))
for path in docs:
    print('-', path.name)

Knowledge base documents: 7
- tip_device_best_practices.txt
- tip_energy_savings.txt
- tip_hvac_optimization.txt
- tip_renewable_integration.txt
- tip_seasonal_energy_management.txt
- tip_smart_home_automation.txt
- tip_storage_optimization.txt


## Knowledge Base Topics

The added documents cover HVAC, automation, renewable integration, seasonal management, and storage optimization.

In [2]:
from pathlib import Path
for path in sorted(Path('data/documents').glob('*.txt')):
    text = path.read_text(encoding='utf-8').strip().replace('\n', ' ')
    print(f'{path.name}: {text[:220]}...')

tip_device_best_practices.txt: Smart appliance scheduling best practices: Run dishwashers, laundry machines, and pool pumps during solar production windows or off-peak pricing periods. Avoid peak evening pricing unless the task is urgent. Use smart pl...
tip_energy_savings.txt: General energy savings: The highest-impact strategy is shifting flexible loads away from expensive grid periods. Combine time-of-use pricing, solar generation forecasts, and device flexibility to reduce cost without redu...
tip_hvac_optimization.txt: HVAC optimization strategies: Pre-cool or pre-heat the home during low-price or high-solar hours before peak electricity periods. During hot afternoons, increase the thermostat setpoint by 2 to 4 degrees during price spi...
tip_renewable_integration.txt: Renewable energy integration: Maximize self-consumption by moving EV charging, water heating, laundry, and dishwashing into mid-day solar windows. If net metering export credits are low, self-consuming solar energy 

## RAG Search Smoke Test

The output below demonstrates that ChromaDB returns relevant documents with source metadata.

In [3]:
from tools import search_energy_tips
result = search_energy_tips.invoke('solar EV charging HVAC storage optimization')
result

{'query': 'solar EV charging HVAC storage optimization',
 'results': [{'id': 'tip_smart_home_automation-0',
   'document': 'Smart home automation tips:\nCreate automations that start flexible appliances when solar production exceeds home load or when electricity prices fall below a target threshold. Use scenes that pause EV charging, pool pumps, and discretionary loads during peak-price alerts.',
   'metadata': {'topic': 'tip_smart_home_automation',
    'chunk_count': 1,
    'chunk_index': 0,
    'source': 'tip_smart_home_automation.txt'},
   'distance': 1.3216978311538696},
  {'id': 'tip_storage_optimization-0',
   'document': 'Energy storage optimization:\nCharge home batteries during solar surplus or low-price overnight hours. Discharge batteries during peak evening pricing while reserving enough capacity for backup needs. Battery optimization should consider round-trip efficiency and the value of avoided grid purchases.',
   'metadata': {'source': 'tip_storage_optimization.txt',
  

In [4]:
for hit in result['results']:
    print(hit['id'], hit['metadata'])
    print(hit['document'][:300].replace('\n', ' '), '\n')

tip_smart_home_automation-0 {'topic': 'tip_smart_home_automation', 'chunk_count': 1, 'chunk_index': 0, 'source': 'tip_smart_home_automation.txt'}
Smart home automation tips: Create automations that start flexible appliances when solar production exceeds home load or when electricity prices fall below a target threshold. Use scenes that pause EV charging, pool pumps, and discretionary loads during peak-price alerts. 

tip_storage_optimization-0 {'source': 'tip_storage_optimization.txt', 'chunk_index': 0, 'chunk_count': 1, 'topic': 'tip_storage_optimization'}
Energy storage optimization: Charge home batteries during solar surplus or low-price overnight hours. Discharge batteries during peak evening pricing while reserving enough capacity for backup needs. Battery optimization should consider round-trip efficiency and the value of avoided grid purchases. 

tip_seasonal_energy_management-0 {'chunk_index': 0, 'source': 'tip_seasonal_energy_management.txt', 'chunk_count': 1, 'topic': 'tip_se